# Cassandra Database Demo

This notebook provides guidance for data operations with Cassandra.

> **Note**: The keyspace `market_data` and tables have been automatically initialized by Docker from the `init.cql` file.

## 1. Connection

Establish a connection to Cassandra and use the keyspace.

In [ ]:
from cassandra.cluster import Cluster
import pandas as pd
from datetime import datetime
import time

while True:
    try:
        # Connect to Cassandra (Match with docker-compose.yml)
        cluster = Cluster(['127.0.0.1'])
        session = cluster.connect()

        # Use the keyspace created by Docker
        session.set_keyspace('market_data')
        
        print("✅ Successfully connected to Cassandra and keyspace 'market_data'!")
        break
    
    except Exception as e:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ⏳ Cassandra is not ready yet (Error: {e}). Retrying in 10 seconds...")
        time.sleep(10)

[21:18:28] ⏳ Cassandra chưa sẵn sàng (Lỗi: ('Unable to connect to any servers', {'127.0.0.1:9042': ConnectionShutdown('Connection to 127.0.0.1:9042 was closed')})). Đang thử lại sau 10 giây...
[21:18:38] ⏳ Cassandra chưa sẵn sàng (Lỗi: ('Unable to connect to any servers', {'127.0.0.1:9042': ConnectionShutdown('Connection to 127.0.0.1:9042 was closed')})). Đang thử lại sau 10 giây...
[21:18:48] ⏳ Cassandra chưa sẵn sàng (Lỗi: ('Unable to connect to any servers', {'127.0.0.1:9042': ConnectionShutdown('Connection to 127.0.0.1:9042 was closed')})). Đang thử lại sau 10 giây...
[21:18:58] ⏳ Cassandra chưa sẵn sàng (Lỗi: ('Unable to connect to any servers', {'127.0.0.1:9042': ConnectionShutdown('Connection to 127.0.0.1:9042 was closed')})). Đang thử lại sau 10 giây...
[21:19:08] ⏳ Cassandra chưa sẵn sàng (Lỗi: ('Unable to connect to any servers', {'127.0.0.1:9042': ConnectionShutdown('Connection to 127.0.0.1:9042 was closed')})). Đang thử lại sau 10 giây...
[21:19:18] ⏳ Cassandra chưa sẵn sàn

## 2. Write Commands

Add sample data to the `klines` table.

In [ ]:
from decimal import Decimal

symbol = 'BTCUSDT'
interval = '1m'
now = datetime.now()
date_bucket = now.date()

insert_query = """
INSERT INTO klines (symbol, interval, date_bucket, timestamp, open, high, low, close, volume)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

session.execute(insert_query, (
    symbol, interval, date_bucket, now, 
    Decimal('65000.50'), Decimal('65100.00'), 
    Decimal('64950.00'), Decimal('65050.25'), 
    Decimal('1.5')
))

print(f"Data written for {symbol}")

Đã ghi dữ liệu cho BTCUSDT


## 3. Query Commands

Read data from Cassandra and display it as a DataFrame.

In [10]:
query = "SELECT * FROM klines WHERE symbol=%s AND interval=%s AND date_bucket=%s LIMIT 5"
rows = session.execute(query, (symbol, interval, date_bucket))

df = pd.DataFrame(list(rows))
df.head()

,symbol,interval,date_bucket,timestamp,close,high,low,open,volume
0,BTCUSDT,1m,2026-05-07,2026-05-07 21:19:38.149,65050.25,65100.0,64950.0,65000.5,1.5


## 4. Advanced

Use Batch to write multiple records at once and set TTL.

In [ ]:
from cassandra.query import BatchStatement

batch = BatchStatement()
prepared_insert = session.prepare(
    "INSERT INTO klines (symbol, interval, date_bucket, timestamp, close) VALUES (?, ?, ?, ?, ?) USING TTL 60"
)

for i in range(3):
    ts = datetime.now()
    batch.add(prepared_insert, (symbol, interval, date_bucket, ts, Decimal(65000 + i)))

session.execute(batch)
print("Batch write with TTL (60s) completed successfully!")

Đã thực hiện ghi Batch với TTL (60s) thành công!
